# Dunnhumby seed 43 — M2+M3에 원형 M4 가중 손실 추가

기존 M2의 사용자 N/V 및 상품 가격 표현과 M3 중심화 가치그래프를 유지하고, **유효성 마스크를 적용한 원형 M4**의 양성행 가중 BPR을 추가합니다. 초기화부터 한 optimizer로 공동학습하는 M5 **한 조건만** 새로 돌립니다. M1·M2·M3·M2+M3·M4는 기존 결과를 읽기만 하며, 원본·선택규칙·입력·M4 가중치가 다르면 학습 전에 중단합니다. M4는 유효행에서 원시가중치 `1+0.5×q_C×상품금액백분위×고객가격대적합도`, 무효행에서는 `1`이며 전체 학습행 평균이 1이 되도록 정규화합니다.

신규상품 추천(학습쌍 제외), `MIN_ITEM_INTER=1`, uniform 음성 K=1, 개발일 684–690만 사용합니다. 최대 300 epoch, 25마다 개발평가, 100 이후 네 번 연속 미개선 시 중단하고 전체 가격·구매금액 가중 적중값@10의 최고 checkpoint를 복원합니다. 주 질문은 **전체** 가격·구매금액 가중 적중값@10과 가중 NDCG@10이 기존 각 모형보다 모두 높아지는지, 정확도 여섯 지표가 M1의 99% 이상인지입니다. 단일 반복노출 개발시드이므로 통과해도 유의성·일반화·CLV 귀속·결합 시너지를 주장할 수 없습니다. 최종 test/holdout은 열지 않습니다.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
from google.colab import drive
ROOT = Path('/content/drive/MyDrive/논문/data')
M2_REPORT = ROOT/'results_v3_dunnhumby_shared_feature_residual_m2_seed43_v1/reports/result.json'
M3_REPORT = ROOT/'results_v3_dunnhumby_clv_m3_centered_value_graph_v1/clv_m3_centered_value_graph_244783a20cc5.json'
JOINT_REPORT = ROOT/'results_v3_dunnhumby_shared_feature_centered_graph_seed43_v1/reports/result.json'
M4_REPORT = ROOT/'results_v3_dunnhumby_linear_nv_original_m4_validity_masked_seed43_v2/reports/result.json'
OUT = ROOT/'results_v3_dunnhumby_shared_feature_centered_graph_original_m4_seed43_v1'
try:
    if not ROOT.is_dir():
        drive.mount('/content/drive')
    if not ROOT.is_dir():
        raise RuntimeError(f'Drive 자료 폴더가 없습니다: {ROOT}')
    for label, path in [('M2', M2_REPORT), ('M3', M3_REPORT), ('M2+M3', JOINT_REPORT), ('M4', M4_REPORT)]:
        if not path.is_file():
            raise RuntimeError(f'{label} 원본 JSON이 없습니다: {path}. 학습은 시작되지 않았습니다.')
except (OSError, ValueError, NotImplementedError) as exc:
    raise RuntimeError('Drive 연결/읽기 실패. 마운트 상태를 확인한 뒤 이 셀부터 다시 실행하세요. 학습은 시작되지 않았습니다.') from exc
SOURCE_COMMIT = '0b040910096751c2d45c83bafe7691e6e1250aa3'
REPO = Path('/content/clv-m2m3m4-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
HEAD = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert HEAD == SOURCE_COMMIT, '코드 버전이 다릅니다.'
for name in ('lightgcn_clv_v3', 'clv_shared_feature_centered_graph_screen', 'clv_linear_nv_original_m4_screen'):
    if name in sys.modules:
        assert Path(sys.modules[name].__file__).resolve().parent == REPO.resolve(), '다른 실험 코드가 로드되었습니다. 런타임을 다시 시작하세요.'
os.chdir(REPO)
sys.path.insert(0, str(REPO))
import clv_shared_feature_centered_graph_original_m4_screen as screen
print('고정 코드:', HEAD)


## 1. 원본·입력·M4 가중치 검증 (GPU 학습 없음)
여기서 오류가 나면 다음 셀을 실행하지 마세요. 기존 모형은 추가 학습하지 않습니다.

In [ ]:
cfg, prepared = screen.prepare(M2_REPORT, M3_REPORT, JOINT_REPORT, M4_REPORT, OUT)
assert cfg.seeds == (43,) and cfg.epochs == 300 and cfg.negative_count == 1
assert len(prepared['anchors']) == 5 and screen._spec()['weighted']
assert prepared['m4_diagnostics']['original_invalid_extra_absent']
print('기준 선택점:', [(a['model_id'], a['selected_epoch'], a['stopped_epoch']) for a in prepared['anchors']])
print('M4 λ:', screen.M4_LAMBDA, '| 정규화 전 평균:', prepared['m4_diagnostics']['train_mean_raw_weight'])
print('M3 엣지 CV:', prepared['graph_audit']['coefficient_of_variation'])
print('새 학습:', screen.MODEL_ID, '| N/V 모두 유지, M4 가중 BPR, 최종 test/holdout 없음')


## 2. 결합 한 조건 학습
GPU가 필요합니다. epoch마다 optimizer·난수상태를 Drive에 저장하므로 중단 시 같은 노트북을 다시 실행해 이어갑니다.

In [ ]:
import torch
assert torch.cuda.is_available(), '학습에는 GPU 런타임을 사용하세요.'
paths = screen.run(cfg, prepared)
print(json.dumps(paths, ensure_ascii=False, indent=2))


## 3. 전체 결과 확인·다운로드
전체/저·중·고CLV의 유리·불리한 지표와 동일 epoch 비교를 함께 확인하세요. ZIP에는 표·진단·판정만, 재개용 checkpoint는 Drive에 남깁니다.

In [ ]:
import pandas as pd
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
report = json.loads((OUT/'reports/result.json').read_text())
assert report['code_version'] == screen.VERSION and report['new_fit_count'] == 1
print('M1 / M2 / M3 / M2+M3 / M4 / M2+M3+M4 전체 절대지표 (독립 선택점)')
print(pd.read_csv(report['paths']['absolute']).set_index('model_id').T.to_string())
print('판정:', json.dumps(report['reading'], ensure_ascii=False, indent=2))
print('전체 비교·같은epoch·곡선·진단:', {k: v for k, v in report['paths'].items() if k != 'absolute'})
zip_path = Path('/content/shared_feature_centered_graph_original_m4_seed43_results.zip')
with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as archive:
    for path in report['paths'].values():
        archive.write(path, arcname=Path(path).name)
files.download(str(zip_path))
